In [ ]:
# pip install xbbg=0.12.3

In [ ]:
import os
import pandas as pd
from xbbg import blp
import time
import numpy as np

In [2]:
import blpapi

# Create a session to the default localhost and port 8194
options = blpapi.SessionOptions()
options.setServerHost("localhost")
options.setServerPort(8194)

session = blpapi.Session(options)

# Attempt to start the session
if session.start():
    print("SUCCESS: blpapi is connected to the Terminal!")
    session.stop() # Cleanly close it
else:
    print("FAILED: blpapi cannot connect. The bbcomm process is likely not running.")

SUCCESS: blpapi is connected to the Terminal!


In [3]:
print("Initiating Bloomberg API connection test...")

try:
    # A simple, universal data point request
    test_data = blp.bdp(tickers='SPX Index', flds=['PX_LAST', 'NAME'])
    
    if test_data.empty:
        print("Status: FAILED. The API is reachable, but returned an empty DataFrame.")
    else:
        print("Status: SUCCESS! Connected to the Terminal.")
        print("-" * 30)
        print(test_data)
        
except Exception as e:
    print(f"Status: FAILED. Cannot reach the API. Error details: {e}")

Initiating Bloomberg API connection test...
Status: SUCCESS! Connected to the Terminal.
------------------------------
                    name  px_last
ticker                           
SPX Index  S&P 500 INDEX  6795.99


C:\Users\est1511\AppData\Local\anaconda3\Lib\site-packages\xbbg\api\reference\reference.py:86: XbbgFutureWarning: xbbg defaults are changing in version 1.0: backend='narwhals' and format='long' will become the new defaults. Set backend/format explicitly to silence this warning. See https://github.com/alpha-xone/xbbg/issues/166 for details.
  return await pipeline.arun(request)


In [6]:
print("--- DIAGNOSTIC 1: The Field Test ---")
# Testing the gold-standard CUR_MKT_CAP field on a universally valid ticker
try:
    test_bdh = blp.bdh(
        tickers=['AAPL US Equity'], 
        flds=['CUR_MKT_CAP'], 
        start_date='2023-01-01', 
        end_date='2023-03-31', 
        Per='M'
    )
    if test_bdh.empty:
        print("FAILED: The BDH call returned empty. Check data limits or field permissions.")
    else:
        print("SUCCESS! The BDH function and CUR_MKT_CAP field are working:")
        print(test_bdh)
except Exception as e:
    print(f"Error during BDH test: {e}")

print("\n--- DIAGNOSTIC 2: The Ticker Format Test ---")
# Let's see exactly what the BDS call is spitting out for your loop
try:
    members = blp.bds('S5INFT Index', 'INDX_MEMBERS', END_DATE_OVERRIDE='20231231')
    tickers = members['member_ticker_and_exchange_code'].tolist()
    formatted_tickers = [f"{t} Equity" for t in tickers[:5]] # Just grab the first 5
    print("If these look weird (e.g., extra spaces or missing country codes), BDH will fail:")
    print(formatted_tickers)
except Exception as e:
    print(f"Error during BDS ticker test: {e}")

--- DIAGNOSTIC 1: The Field Test ---
SUCCESS! The BDH function and CUR_MKT_CAP field are working:
ticker     AAPL US Equity
field         cur_mkt_cap
date                     
2023-01-31   2.285007e+06
2023-02-28   2.332313e+06
2023-03-31   2.609039e+06

--- DIAGNOSTIC 2: The Ticker Format Test ---
If these look weird (e.g., extra spaces or missing country codes), BDH will fail:
['AAPL UW Equity', 'ACN UN Equity', 'ADBE UW Equity', 'ADI UW Equity', 'ADSK UW Equity']


### HHI extraction ($HHI_{j,t}$) + Bayesian Trigger ($Cascade_{j,t}$)

In [ ]:
# 1. Define the master dictionary using the underlying S&P 500 Sector Indices
spdr_indices = {
    'Technology': 'S5INFT Index',
    'Health Care': 'S5HLTH Index',
    'Energy': 'S5ENRS Index',
    'Financials': 'S5FINL Index',
    'Industrials': 'S5INDU Index',
    'Consumer Staples': 'S5CONS Index',
    'Consumer Discretionary': 'S5COND Index',
    'Utilities': 'S5UTIL Index',
    'Materials': 'S5MATR Index',
    'Real Estate': 'S5RLST Index',
    'Communication': 'S5TELS Index'
}

start_date = '2005-01-31'
end_date = '2026-02-28'
sample_dates = pd.date_range(start=start_date, end=end_date, freq='YE')

cascade_frames = []

# 2. Loop through each sector index
for sector_name, index_ticker in spdr_indices.items():
    print(f"Processing {sector_name} ({index_ticker})...")
    historical_tickers = set()

    # Get historical constituents from the Index
    for date in sample_dates:
        date_str = date.strftime('%Y%m%d')
        try:
            members = blp.bds(index_ticker, 'INDX_MEMBERS', END_DATE_OVERRIDE=date_str)
            if not members.empty:
                tickers = members['member_ticker_and_exchange_code'].tolist()
                formatted_tickers = [f"{t} Equity" for t in tickers]
                historical_tickers.update(formatted_tickers)
            time.sleep(0.5) 
        except Exception as e:
            pass 
            
    master_ticker_list = list(historical_tickers)
    
    # Skip if no tickers found
    if not master_ticker_list:
        print(f"No constituents found for {sector_name}. Skipping.\n")
        continue

    # Pull monthly market cap for the constituents
    mkt_cap_data = blp.bdh(
        tickers=master_ticker_list,
        flds=['CUR_MKT_CAP'],
        start_date=start_date,
        end_date=end_date,
        Per='M'
    )
    
    if mkt_cap_data.empty:
        print('empty')
        continue

    mkt_cap_data.columns = mkt_cap_data.columns.get_level_values(0)
    mkt_cap_data.fillna(0, inplace=True)

    # Calculate HHI
    total_sector_mkt_cap = mkt_cap_data.sum(axis=1)
    market_shares = mkt_cap_data.div(total_sector_mkt_cap, axis=0)
    squared_shares = market_shares ** 2
    hhi_series = squared_shares.sum(axis=1)

    hhi_df = pd.DataFrame({
        'Date': hhi_series.index,
        'Sector': sector_name,
        'Index_Ticker': index_ticker,
        'HHI': hhi_series.values
    })

    # 3. Export to CSV automatically
    filename = f"{sector_name.replace(' ', '_')}_HHI.csv"
    hhi_df.to_csv(filename, index=False)
    print(f"Saved {filename} to {os.path.abspath(filename)}\n")

    # --- Calculate the cascade proxy for this sector --

    # 1. Identify all unique tickers that have EVER been in the Top 3 for this sector
    # We apply a lambda function to grab the index (tickers) of the 3 largest values per row
    top_3_per_month = mkt_cap_data.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)

    # Flatten the list and get unique tickers
    unique_top_tickers = list(set([ticker for month_list in top_3_per_month for ticker in month_list]))
    print(f"Total unique top-3 constituents identified: {len(unique_top_tickers)}")

    # 2. Pull 30-day realized volatility for only these heavyweights
    vol_data = blp.bdh(
        tickers=unique_top_tickers,
        flds=['VOLATILITY_30D'],
        start_date=start_date,
        end_date=end_date,
        Per='M'
    )

    # Flatten columns
    if not vol_data.empty:
        vol_data.columns = vol_data.columns.get_level_values(0)

        # 3. Calculate the rolling 2-Standard-Deviation threshold and create the dummy variable
        cascade_flags = pd.DataFrame(index=vol_data.index)

        for ticker in vol_data.columns:
            # Calculate a rolling 12-month mean and standard deviation for each stock
            rolling_mean = vol_data[ticker].rolling(window=12).mean()
            rolling_std = vol_data[ticker].rolling(window=12).std()
            
            # Define the threshold
            spike_threshold = rolling_mean + (2 * rolling_std)
            
            # Create binary flag: 1 if volatility > threshold, 0 otherwise
            cascade_flags[ticker] = np.where(vol_data[ticker] > spike_threshold, 1, 0)

        # If ANY of the top 3 stocks in a given month triggered the cascade, flag the sector
        sector_cascade = cascade_flags.max(axis=1)

        temp_cascade_df = pd.DataFrame({
            'Date': sector_cascade.index,
            'Sector': sector_name,  # Maps the trigger back to the specific sector
            'Cascade_Trigger': sector_cascade.values
        })
        
        # Append to our master list
        cascade_frames.append(temp_cascade_df)

if cascade_frames:
    master_cascade_df = pd.concat(cascade_frames, ignore_index=True)
    master_cascade_df.to_csv("Sector_Cascade_Proxy.csv", index=False)
    print("Master Cascade Proxy calculated and saved for all sectors.")

print("All sectors processed and exported.")

Processing Technology (S5INFT Index)...
Saved Technology_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Technology_HHI.csv

Processing Health Care (S5HLTH Index)...
Saved Health_Care_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Health_Care_HHI.csv

Processing Energy (S5ENRS Index)...
Saved Energy_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Energy_HHI.csv

Processing Financials (S5FINL Index)...
Saved Financials_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Financials_HHI.csv

Processing Industrials (S5INDU Index)...
Saved Industrials_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Industrials_HHI.csv

Processing Consumer Staples (S5CONS Index)...
Saved Consumer_Staples_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Consumer_Staples_HHI.csv

Processing Consumer Discretionary (S5COND Index)...
Saved Consumer_Discretionary_HHI.csv to C:\Users\est1511\Documents\Econ Thesis\Consumer_Discretionary_HHI.csv

Processing Utilities (S5UTIL Index)...
Saved Utilities_HHI.csv 

### Hub_Shock

In [ ]:
"""
Modified data collection: Hub_Shock with rolling 36-month top-3 window.

Changes from the original Cascade_Trigger block:
  1. Intermediate market cap and volatility dataframes are now saved to disk
     per sector, so Phase 2 (Hub_Shock construction) can be re-run without
     re-querying Bloomberg if the rule changes again.
  2. Top-3 heavyweight constituents are identified using a rolling 36-month
     window rather than the union over the full sample, addressing the
     feedback that the trigger should reflect *currently* central nodes.
  3. Variable renamed from Cascade_Trigger to Hub_Shock, per Walker's note
     that the binary flags volatility spikes in heavyweight firms rather
     than propagating cascade failures.
  4. The 12-month rolling z-score spike rule (vol > mean + 2*std) is
     unchanged. Only the heavyweight-identification step uses the rolling
     36-month window.
"""

import os
import time
import numpy as np
import pandas as pd
from xbbg import blp

# --- Configuration ---
WINDOW_MONTHS = 36          # rolling window for identifying heavyweight constituents
SPIKE_LOOKBACK = 12         # rolling window for vol z-score (unchanged)
SPIKE_THRESHOLD_SD = 2      # number of std deviations above mean (unchanged)
INTERMEDIATES_DIR = './Data/Intermediates'
os.makedirs(INTERMEDIATES_DIR, exist_ok=True)

# --- Sector index mapping (unchanged) ---
spdr_indices = {
    'Technology': 'S5INFT Index',
    'Health Care': 'S5HLTH Index',
    'Energy': 'S5ENRS Index',
    'Financials': 'S5FINL Index',
    'Industrials': 'S5INDU Index',
    'Consumer Staples': 'S5CONS Index',
    'Consumer Discretionary': 'S5COND Index',
    'Utilities': 'S5UTIL Index',
    'Materials': 'S5MATR Index',
    'Real Estate': 'S5RLST Index',
    'Communication': 'S5TELS Index'
}

start_date = '2005-01-31'
end_date = '2026-02-28'
sample_dates = pd.date_range(start=start_date, end=end_date, freq='YE')


# ============================================================
# PHASE 1: Bloomberg pulls -> save intermediates
# ============================================================
# Run this once. If you later need to revise the Hub_Shock rule, you can
# skip Phase 1 entirely and re-run only Phase 2 from the saved intermediates.

for sector_name, index_ticker in spdr_indices.items():
    print(f"Processing {sector_name} ({index_ticker})...")

    safe_name = sector_name.replace(' ', '_')
    mkt_cap_path = os.path.join(INTERMEDIATES_DIR, f"{safe_name}_mkt_cap.parquet")
    vol_path = os.path.join(INTERMEDIATES_DIR, f"{safe_name}_vol_30d.parquet")

    # --- Historical constituents ---
    historical_tickers = set()
    for date in sample_dates:
        date_str = date.strftime('%Y%m%d')
        try:
            members = blp.bds(index_ticker, 'INDX_MEMBERS', END_DATE_OVERRIDE=date_str)
            if not members.empty:
                tickers = members['member_ticker_and_exchange_code'].tolist()
                historical_tickers.update([f"{t} Equity" for t in tickers])
            time.sleep(0.5)
        except Exception:
            pass

    master_ticker_list = list(historical_tickers)
    if not master_ticker_list:
        print(f"  No constituents found for {sector_name}. Skipping.")
        continue

    # --- Monthly market cap ---
    mkt_cap_data = blp.bdh(
        tickers=master_ticker_list,
        flds=['CUR_MKT_CAP'],
        start_date=start_date,
        end_date=end_date,
        Per='M'
    )
    if mkt_cap_data.empty:
        print(f"  Empty market cap data for {sector_name}. Skipping.")
        continue

    mkt_cap_data.columns = mkt_cap_data.columns.get_level_values(0)
    mkt_cap_data.fillna(0, inplace=True)
    mkt_cap_data.to_parquet(mkt_cap_path)

    # --- HHI (unchanged from original) ---
    total_sector_mkt_cap = mkt_cap_data.sum(axis=1)
    market_shares = mkt_cap_data.div(total_sector_mkt_cap, axis=0)
    hhi_series = (market_shares ** 2).sum(axis=1)

    hhi_df = pd.DataFrame({
        'Date': hhi_series.index,
        'Sector': sector_name,
        'Index_Ticker': index_ticker,
        'HHI': hhi_series.values
    })
    hhi_df.to_csv(f"{safe_name}_HHI.csv", index=False)

    # --- Vol-pull universe ---
    # Note: "ever top-3 in any 36-month window" equals "ever top-3 in any single
    # month" equals "ever top-3 in the full sample," so the vol-pull universe
    # is unchanged. The rolling logic is applied later in Phase 2.
    top_3_per_month = mkt_cap_data.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)
    unique_top_tickers = list({t for month_list in top_3_per_month for t in month_list})

    # --- 30-day realized vol for heavyweights ---
    vol_data = blp.bdh(
        tickers=unique_top_tickers,
        flds=['VOLATILITY_30D'],
        start_date=start_date,
        end_date=end_date,
        Per='M'
    )
    if vol_data.empty:
        print(f"  Empty vol data for {sector_name}. Skipping vol save.")
        continue

    vol_data.columns = vol_data.columns.get_level_values(0)
    vol_data.to_parquet(vol_path)
    print(f"  Saved intermediates for {sector_name}.")


# ============================================================
# PHASE 2: Hub_Shock construction (offline, fully re-runnable)
# ============================================================

hub_shock_frames = []

for sector_name in spdr_indices.keys():
    safe_name = sector_name.replace(' ', '_')
    mkt_cap_path = os.path.join(INTERMEDIATES_DIR, f"{safe_name}_mkt_cap.parquet")
    vol_path = os.path.join(INTERMEDIATES_DIR, f"{safe_name}_vol_30d.parquet")

    if not (os.path.exists(mkt_cap_path) and os.path.exists(vol_path)):
        print(f"Missing intermediates for {sector_name}, skipping Hub_Shock.")
        continue

    mkt_cap_data = pd.read_parquet(mkt_cap_path)
    vol_data = pd.read_parquet(vol_path)
    mkt_cap_data.index = pd.to_datetime(mkt_cap_data.index)
    vol_data.index = pd.to_datetime(vol_data.index)

    # --- Rolling 36-month top-3 set per month ---
    # For each month t, the heavyweight universe is the union of top-3
    # constituents over months [t - (WINDOW_MONTHS - 1), t].
    rolling_top3_per_month = {}
    dates = mkt_cap_data.index.sort_values()
    for i, t in enumerate(dates):
        window_start_idx = max(0, i - (WINDOW_MONTHS - 1))
        window = mkt_cap_data.iloc[window_start_idx:i + 1]
        top_set = set()
        for _, row in window.iterrows():
            top_set.update(row.nlargest(3).index.tolist())
        rolling_top3_per_month[t] = top_set

    # --- Vol spike flags (12-month rolling z-score, unchanged) ---
    rolling_mean = vol_data.rolling(window=SPIKE_LOOKBACK).mean()
    rolling_std = vol_data.rolling(window=SPIKE_LOOKBACK).std()
    spike_threshold = rolling_mean + SPIKE_THRESHOLD_SD * rolling_std
    spike_flags = (vol_data > spike_threshold).astype(int)

    # --- Hub_Shock: 1 if any rolling-top-3 ticker spiked this month ---
    hub_shock = pd.Series(0, index=mkt_cap_data.index, dtype=int)
    for t, ticker_set in rolling_top3_per_month.items():
        if t not in spike_flags.index:
            continue
        relevant = [tk for tk in ticker_set if tk in spike_flags.columns]
        if relevant:
            hub_shock.loc[t] = int(spike_flags.loc[t, relevant].max())

    sector_df = pd.DataFrame({
        'Date': hub_shock.index,
        'Sector': sector_name,
        'Hub_Shock': hub_shock.values
    })
    hub_shock_frames.append(sector_df)
    print(f"  Hub_Shock for {sector_name}: {int(hub_shock.sum())} flagged months "
          f"out of {len(hub_shock)}.")

# --- Save master Hub_Shock file ---
if hub_shock_frames:
    master_hub_shock = pd.concat(hub_shock_frames, ignore_index=True)
    master_hub_shock.to_csv("Sector_Hub_Shock.csv", index=False)
    print(f"\nSaved Sector_Hub_Shock.csv with {len(master_hub_shock)} rows.")

Processing Technology (S5INFT Index)...
  Saved intermediates for Technology.
Processing Health Care (S5HLTH Index)...
  Saved intermediates for Health Care.
Processing Energy (S5ENRS Index)...
  Saved intermediates for Energy.
Processing Financials (S5FINL Index)...
  Saved intermediates for Financials.
Processing Industrials (S5INDU Index)...
  Saved intermediates for Industrials.
Processing Consumer Staples (S5CONS Index)...
  Saved intermediates for Consumer Staples.
Processing Consumer Discretionary (S5COND Index)...
  Saved intermediates for Consumer Discretionary.
Processing Utilities (S5UTIL Index)...
  Saved intermediates for Utilities.
Processing Materials (S5MATR Index)...
  Saved intermediates for Materials.
Processing Real Estate (S5RLST Index)...
  Saved intermediates for Real Estate.
Processing Communication (S5TELS Index)...
  Saved intermediates for Communication.
  Hub_Shock for Technology: 33 flagged months out of 254.
  Hub_Shock for Health Care: 32 flagged months o

### Sector Controls ($X_{j,t}$)

In [16]:
# 1. Define the ETF dictionary (Targeting the tradable assets)
spdr_etfs = {
    'Technology': 'XLK US Equity',
    'Health Care': 'XLV US Equity',
    'Energy': 'XLE US Equity',
    'Financials': 'XLF US Equity',
    'Industrials': 'XLI US Equity',
    'Consumer Staples': 'XLP US Equity',
    'Consumer Discretionary': 'XLY US Equity',
    'Utilities': 'XLU US Equity',
    'Materials': 'XLB US Equity',
    'Real Estate': 'XLRE US Equity',
    'Communication': 'XLC US Equity'
}

# 2. Extract the tickers into a list
etf_tickers = list(spdr_etfs.values())

# 3. Pull the fundamental and performance controls
sector_controls = blp.bdh(
    tickers=etf_tickers,
    flds=['TOT_RETURN_INDEX_GROSS_DVDS', 'PE_RATIO'],
    start_date=start_date, # Assuming '2005-01-31' is still in memory
    end_date=end_date,     # Assuming '2023-12-31' is still in memory
    Per='M'
)

# Flatten columns if necessary, then save
sector_controls.columns = sector_controls.columns.map('_'.join) # Flattens multi-index cleanly
sector_controls.to_csv("Sector_Controls.csv")
print("Sector controls downloaded and saved.")

Sector controls downloaded and saved.


### 25-Delta Risk Reversal ($RR_{j,t}$)
Use OptionMetrics Ivy DB; waiting for approval.

### Option Chains for MFIS ($MFIS_{j,t}$)
Use OptionMetrics Ivy DB; waiting for approval.

### Realized Skewness ($RSKEW_{j,t}$)

In [31]:
print("Pulling daily returns for Realized Tail Risk metrics...")

# Pull DAILY returns for the ETFs
daily_data = blp.bdh(
    tickers=etf_tickers,
    flds=['DAY_TO_DAY_TOT_RETURN_GROSS_DVDS'],
    start_date=start_date, 
    end_date=end_date,     
    Per='D'                
)

realized_risk_frames = []

if not daily_data.empty:
    # Flatten columns and convert index to Datetime
    daily_data.columns = daily_data.columns.get_level_values(0)
    daily_data.index = pd.to_datetime(daily_data.index)

    for ticker in etf_tickers:
        if ticker in daily_data.columns:
            # Extract the daily return series
            daily_returns = daily_data[ticker].dropna() / 100.0 
            
            # 1. Calculate Realized Skewness (Monthly)
            # FIX: Use .apply() to pass the skew function to the resampled bins
            monthly_skew = daily_returns.resample('ME').apply(lambda x: x.skew())
            
            # 2. Calculate Downside Realized Volatility (Monthly)
            downside_returns = daily_returns[daily_returns < 0]
            # FIX: Conditionally handle months with zero negative days inside the apply function
            monthly_drv = downside_returns.resample('ME').apply(
                lambda x: np.sqrt((x**2).sum()) if not x.empty else 0.0
            )
            
            # Ensure the indices align perfectly before building the DataFrame
            temp_df = pd.DataFrame({
                'Date': monthly_skew.index,
                'Ticker': ticker,
                'Realized_Skewness': monthly_skew.values,
                'Downside_Volatility': monthly_drv.reindex(monthly_skew.index, fill_value=0.0).values
            })
            realized_risk_frames.append(temp_df)

if realized_risk_frames:
    realized_panel = pd.concat(realized_risk_frames, ignore_index=True)
    realized_panel.to_csv("Realized_Tail_Risk_Placeholders.csv", index=False)
    print("Realized tail risk metrics successfully calculated and saved.")
else:
    print("FAILED: No daily return data was found.")

Pulling daily returns for Realized Tail Risk metrics...
Realized tail risk metrics successfully calculated and saved.


### Macroeconomic Controls (Out-of-Sample Baselines)

In [27]:
import pandas_datareader.data as web
import datetime

# Define your date range using standard datetime objects
start = datetime.datetime(2005, 1, 1)
end = datetime.datetime(2026, 2, 28)

# These are the official FRED API mnemonics
fred_tickers = [
    'VIXCLS', # CBOE Volatility Index (Daily)
    'GS10',   # 10-Year Treasury Constant Maturity Rate (Monthly)
    'GS2',    # 2-Year Treasury Constant Maturity Rate (Monthly)
    'BAA',    # Moody's Seasoned Baa Corporate Bond Yield (Monthly)
    'AAA'     # Moody's Seasoned Aaa Corporate Bond Yield (Monthly)
]

# Pull the data directly from the Federal Reserve
fred_data = web.DataReader(fred_tickers, 'fred', start, end)

# Because VIX is daily and the others are monthly, we resample to a clean end-of-month frequency
monthly_macro = fred_data.resample('ME').last()

# Calculate the spreads
final_macro_df = pd.DataFrame(index=monthly_macro.index)
final_macro_df['VIX'] = monthly_macro['VIXCLS']
final_macro_df['Term_Spread'] = monthly_macro['GS10'] - monthly_macro['GS2']
final_macro_df['Credit_Spread'] = monthly_macro['BAA'] - monthly_macro['AAA']

final_macro_df.dropna(inplace=True)
final_macro_df.to_csv("Macro_Controls_FRED.csv")
print("Macroeconomic controls successfully pulled from FRED and saved.")

Macroeconomic controls successfully pulled from FRED and saved.


### I/O Tables ($SupplierHHI_{j,t}$)
Use BEA website